In [4]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.utils import resample
from sklearn.base import clone

# ----------------------------
# 1) Cargar y preparar datos
# ----------------------------
data = pd.read_csv('Default.csv')

# Binarias 0/1
data['default'] = (data['default'] == 'Yes').astype(int)
data['student'] = (data['student'] == 'Yes').astype(int)

# Features candidatas y target
feature_names = ['balance', 'income', 'student']
y = data['default'].values
X_all = data[feature_names].values

# Copia para bootstrap
df_boot = data.copy(deep=True)

# ----------------------------
# 2) Modelo original con 3 X
# ----------------------------
base_model = LogisticRegression(max_iter=1000)
base_model.fit(X_all, y)

b0_orig = float(base_model.intercept_)
b_orig = dict(zip(feature_names, base_model.coef_[0]))

print("Modelo original (con 3 X)")
print(f"  β0 (intercepto): {b0_orig:.6f}")
for k in feature_names:
    print(f"  β({k}): {b_orig[k]:.8f}")



Modelo original (con 3 X)
  β0 (intercepto): -2.950850
  β(balance): 0.00408201
  β(income): -0.00013389
  β(student): -3.89009045


In [8]:
from sklearn.metrics import accuracy_score

n_boot = 5000
n_samples_boot = 5000
rng = np.random.RandomState(42)

coef_mat = np.full((n_boot, 1 + len(feature_names)), np.nan)
acc_list = []

for i in range(n_boot):
    sample = resample(data, replace=True, n_samples=n_samples_boot, random_state=i)
    cols = rng.choice(feature_names, size=2, replace=False)
    Xb = sample[cols].values
    yb = sample['default'].values

    m = clone(base_model)
    m.fit(Xb, yb)

    coef_mat[i, 0] = m.intercept_[0]
    for j, col in enumerate(cols):
        idx = 1 + feature_names.index(col)
        coef_mat[i, idx] = m.coef_[0, j]

    
    X_full = data[cols].values
    y_full = data['default'].values
    y_pred = m.predict(X_full)
    acc = accuracy_score(y_full, y_pred)
    acc_list.append(acc)

coef_df = pd.DataFrame(coef_mat, columns=['intercepto'] + feature_names)
agg_mean = coef_df.mean(skipna=True)
agg_std = coef_df.std(skipna=True, ddof=1)

# --- 4. Accuracy promedio del bootstrap ---
acc_array = np.array(acc_list)
acc_mean = acc_array.mean()
acc_std = acc_array.std(ddof=1)
acc_ic95 = (np.percentile(acc_array, 2.5), np.percentile(acc_array, 97.5))

print("\nResultados Bootstrap (5000 remuestreos con 2 columnas al azar):")
print(f"  Accuracy medio: {acc_mean:.4f}")
print(f"  Desv. estándar: {acc_std:.4f}")
print(f"  IC95%: [{acc_ic95[0]:.4f}, {acc_ic95[1]:.4f}]")

# --- 5. Modelo normal (con las 3 columnas completas) ---
base_model.fit(data[feature_names].values, data['default'].values)
y_pred_full = base_model.predict(data[feature_names].values)
acc_normal = accuracy_score(data['default'].values, y_pred_full)

print("\nModelo normal (3 columnas completas):")
print(f"  Accuracy: {acc_normal:.4f}")

# --- 6. Comparación ---
print("\nComparación final:")
print(f"  Accuracy promedio Bootstrap: {acc_mean:.4f}")
print(f"  Accuracy Modelo Completo    : {acc_normal:.4f}")



Resultados Bootstrap (5000 remuestreos con 2 columnas al azar):
  Accuracy medio: 0.9698
  Desv. estándar: 0.0034
  IC95%: [0.9659, 0.9737]

Modelo normal (3 columnas completas):
  Accuracy: 0.9671

Comparación final:
  Accuracy promedio Bootstrap: 0.9698
  Accuracy Modelo Completo    : 0.9671
